In [19]:
import fitz
import spacy
import json
import requests
import os
import re
import torch

from datetime import datetime
from pathlib import Path
from transformers import AutoModel, AutoTokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pymilvus import (
    connections,
    FieldSchema, CollectionSchema, DataType,
    Collection
)

In [20]:
import pymilvus
print(pymilvus.__version__)


2.6.1


In [21]:
from pymilvus import connections
connections.connect("default", host="127.0.0.1", port="19530")
print(connections.has_connection("default"))  # True


True


In [22]:

# ========================
# CONFIGURAÇÃO DO MILVUS
# ========================
connections.connect("default", host="127.0.0.1", port="19530")

# Nome da coleção no Milvus
COLLECTION_NAME = "rag_embeddings_milvus"

# Esquema da coleção
fields = [
    FieldSchema(name="chunk_id", dtype=DataType.VARCHAR, max_length=200, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="source_file", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="source_url", dtype=DataType.VARCHAR, max_length=1000),
    FieldSchema(name="chunk_index", dtype=DataType.INT64),
    FieldSchema(name="chunk_text", dtype=DataType.VARCHAR, max_length=6000),
]

schema = CollectionSchema(fields, description="RAG embeddings with Milvus")

from pymilvus import utility

if utility.has_collection(COLLECTION_NAME):
    collection = Collection(COLLECTION_NAME)
else:
    collection = Collection(name=COLLECTION_NAME, schema=schema)


# ========================
# NLP e utilitários
# ========================
download_dir = './temp_pdfs/'
os.makedirs(download_dir, exist_ok=True)

nlp = spacy.load("pt_core_news_sm")
nlp.max_length = 2_000_000

GENERIC_IGNORE_KEYWORDS = {
    "sumário", "índice", "resumo", "anexo", "figura", "tabela", "referência", "bibliografia",
    "conteúdo", "protocolo", "gov", "secretaria", "assinatura", "documento assinado",
    "orientações gerais", "preenchimento", "manual", "parecer técnico"
}  
PHRASE_BLACKLIST = [
    r"monitoramento de.*\(e-cenários\)",
    r"^página \d+.*",
    r"^mapas\s*-\s*",
    r"^inserção do estudo.*",
    r"^um arquivo não substitui.*",
    r"^o link para.*tipologia.*",
    r"documentos, manifestações.*",
    r"requerimento.*preenchido.*",
    r"documento.*válido.*"
]
KEY_VERBS = {
    "avaliar", "caracterizar", "delimitar", "analisar", "estimar",
    "descrever", "propor", "identificar", "considerar", "indicar",
    "demonstrar", "impactar", "recomendar", "quantificar"
}

def download_pdf(url, download_dir):
    response = requests.get(url)
    if response.status_code == 200:
        raw_filename = url.split("/")[-1]
        safe_filename = re.sub(r'[\\/*?:"<>|]', "_", raw_filename)
        filename = os.path.join(download_dir, safe_filename)
        with open(filename, 'wb') as file:
            file.write(response.content)
        return filename
    return None

def contains_key_verb(sent):
    return any(tok.lemma_ in KEY_VERBS for tok in nlp(sent))

def is_line_irrelevant(line):
    lower = line.lower()
    if len(lower.strip()) < 15:
        return True
    if any(k in lower for k in GENERIC_IGNORE_KEYWORDS):
        return True
    if any(re.search(pattern, lower) for pattern in PHRASE_BLACKLIST):
        return True
    if re.search(r"\d{2}/\d{2}/\d{2,4}", lower):
        return True
    if lower.endswith(".pdf"):
        return True
    if re.match(r"^\d+(\.\d+)+\s+", lower):
        return True
    if lower.isupper() and len(lower.split()) > 5:
        return True
    return False


def split_text(text, max_chars=500000):
    chunks = []
    while len(text) > max_chars:
        split_point = text[:max_chars].rfind(".") + 1  # tenta dividir no ponto final
        if split_point < 50:
            split_point = max_chars  # se não encontrar ponto, quebra direto
        chunks.append(text[:split_point].strip())
        text = text[split_point:].strip()
    if text:
        chunks.append(text)
    return chunks

def extract_relevant_sentences(text, min_words=4):
    relevant = []
    chunks = split_text(text)
    for chunk in chunks:
        doc = nlp(chunk)
        for sent in doc.sents:
            s = sent.text.strip()
            if len(s.split()) < min_words:
                continue
            if contains_key_verb(s):
                relevant.append(s)
            elif any(tok.pos_ == "VERB" for tok in sent):
                relevant.append(s)
    return relevant

def extract_clean_text_with_spacy(pdf_path, source_url):
    try:
        with fitz.open(pdf_path) as doc:
            raw_text = "\n".join([page.get_text("text") for page in doc])
        lines = raw_text.splitlines()
        filtered = [line.strip() for line in lines if not is_line_irrelevant(line.strip())]
        cleaned_text = " ".join(filtered)
        relevant_sentences = extract_relevant_sentences(cleaned_text)
        return {
            "source_url": source_url,
            "text": "\n".join(relevant_sentences) if relevant_sentences else None
        }
    except Exception as e:
        print(f"Erro ao processar {pdf_path}: {e}")
        return None








In [23]:
# ========================
# EMBEDDINGS
# ========================
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def split_text_into_chunks(text, chunk_size=2000, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    return splitter.split_text(text)

def get_embedding(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy().tolist()

In [24]:
# ========================
# CRIAR ÍNDICE NO CAMPO DE EMBEDDINGS
# ========================
index_params = {
    "index_type": "IVF_FLAT",   # você pode trocar para "HNSW", "IVF_SQ8", etc.
    "metric_type": "IP",        # ou "L2" se preferir distância euclidiana
    "params": {"nlist": 1024}   # parâmetro do índice
}

if not collection.has_index():
    print("🔧 Criando índice para embeddings...")
    collection.create_index(field_name="embedding", index_params=index_params)
    print("✅ Índice criado com sucesso!")


In [25]:
# ========================
# CARREGAR E INSERIR PDFs
# ========================
document_urls = [
    "https://cetesb.sp.gov.br/eiarima/eia/EIA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf",
    "https://cetesb.sp.gov.br/eiarima/rima/RIMA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf",
    "https://cetesb.sp.gov.br/eiarima/eia/EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf"
]

for url in document_urls:
    print(f"Baixando: {url}...")
    pdf_path = download_pdf(url, download_dir)
    if pdf_path:
        pdf_data = extract_clean_text_with_spacy(pdf_path, url)
        if pdf_data and isinstance(pdf_data.get("text"), str):
            chunks = split_text_into_chunks(pdf_data["text"])
            embeddings = [get_embedding(c) for c in chunks]

            data = [
                [f"{os.path.basename(pdf_path)}_chunk_{i}" for i in range(len(chunks))],
                embeddings,
                [os.path.basename(pdf_path)] * len(chunks),
                [url] * len(chunks),
                list(range(len(chunks))),
                chunks
            ]

            collection.insert(data)
            print(f"✅ Documento {pdf_path} salvo com {len(chunks)} chunks.")

Baixando: https://cetesb.sp.gov.br/eiarima/eia/EIA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf...
✅ Documento ./temp_pdfs/EIA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf salvo com 602 chunks.
Baixando: https://cetesb.sp.gov.br/eiarima/rima/RIMA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf...
✅ Documento ./temp_pdfs/RIMA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf salvo com 35 chunks.
Baixando: https://cetesb.sp.gov.br/eiarima/eia/EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf...
✅ Documento ./temp_pdfs/EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf salvo com 562 chunks.


In [26]:
# ========================
# FUNÇÃO DE BUSCA
# ========================
def test_query(query: str, top_k: int = 5):
    q_emb = get_embedding(query)

    collection.load()
    results = collection.search(
        data=[q_emb],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
    )

    print("\n🔎 Resultados da busca:")
    for idx, r in enumerate(results[0]):
        print(f"\nResultado {idx+1}:")
        print(f"Score: {r.score:.4f}")
        print(f"Documento: {r.entity.get('source_file')}")
        print(f"Chunk Index: {r.entity.get('chunk_index')}")
        print(f"Link: {r.entity.get('source_url')}")
        print(f"Texto: {r.entity.get('chunk_text')}")

# Exemplo de consulta
test_query("quais árvores existem em são paulo?")


🔎 Resultados da busca:

Resultado 1:
Score: 5.3813
Documento: EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf
Chunk Index: 169
Link: https://cetesb.sp.gov.br/eiarima/eia/EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf
Texto: Tel: + 55 (19)
Família Scolopacidae Gallinago paraguaiae Tringa solitaria maçarico-solitário
Leptotila verreauxi Leptotila rufaxilla juriti-de-testa-branca
Tel: + 55 (19)
Strigidae Megascops choliba corujinha-do-mato Pulsatrix koeniswaldiana murucututu-de-barriga-amarela
Athene cunicularia coruja-buraqueira coruja-orelhuda mocho-dos-banhados
Tel: + 55 (19)
Hydropsalis parvula Hydropsalis torquata bacurau-tesoura
nigricollis beija-flor-de-veste-preta Chlorostilbon lucidus besourinho-de-bico-vermelho
Tel: + 55 (19)
Amazilia lactea beija-flor-de-peito-azul Heliomaster longirostris bico-reto-cinzento
torquata martim-pescador-grande Chloroceryle amazona martim-pescador-verde
Família Momotidae Baryphthengus ruficapillus Ordem Galbuliformes Família Ga

In [27]:

# from pymilvus import utility
# COLLECTION_NAME = "rag_embeddings_milvus"
# connections.connect("default", host="127.0.0.1", port="19530")
# utility.drop_collection(COLLECTION_NAME)
